# Week 5 - Model Optimization and Experimentation

## Breast Cancer Diagnostic Classification

This notebook performs hyperparameter optimization using:

- GridSearchCV for Logistic Regression
- RandomizedSearchCV for Random Forest

ROC-AUC is used as the primary optimization metric.


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold
)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

print("Libraries imported successfully.")


## 2. Load Dataset and Split Data

In [ ]:
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))


## 3. Cross-Validation Strategy

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("5-Fold Stratified Cross-Validation configured.")


## 4. Logistic Regression - Grid Search

The parameter grid explores:

- C
- Solver
- Class weight

GridSearchCV evaluates every parameter combination.


In [ ]:
logistic_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    (
        "model",
        LogisticRegression(
            max_iter=5000,
            random_state=42
        )
    )
])

logistic_parameters = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__solver": ["liblinear", "lbfgs"],
    "model__class_weight": [None, "balanced"]
}

logistic_search = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=logistic_parameters,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

logistic_search.fit(X_train, y_train)

print("Best Logistic Regression Parameters:")
print(logistic_search.best_params_)

print(
    f"Best Cross-Validation ROC-AUC: "
    f"{logistic_search.best_score_:.4f}"
)


## 5. Random Forest - Randomized Search

RandomizedSearchCV evaluates a selected number of parameter
combinations from the defined distributions.


In [ ]:
random_forest = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

forest_parameters = {
    "n_estimators": [100, 200, 300, 400, 500],
    "max_depth": [None, 5, 10, 15, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", None],
    "class_weight": [None, "balanced"]
}

forest_search = RandomizedSearchCV(
    estimator=random_forest,
    param_distributions=forest_parameters,
    n_iter=20,
    scoring="roc_auc",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    return_train_score=True
)

forest_search.fit(X_train, y_train)

print("Best Random Forest Parameters:")
print(forest_search.best_params_)

print(
    f"Best Cross-Validation ROC-AUC: "
    f"{forest_search.best_score_:.4f}"
)


## 6. Optimization Results

In [ ]:
optimization_results = pd.DataFrame([
    {
        "model": "Logistic Regression",
        "optimization_method": "Grid Search",
        "best_cv_roc_auc": logistic_search.best_score_,
        "best_parameters": str(logistic_search.best_params_)
    },
    {
        "model": "Random Forest",
        "optimization_method": "Randomized Search",
        "best_cv_roc_auc": forest_search.best_score_,
        "best_parameters": str(forest_search.best_params_)
    }
])

display(optimization_results)


## 7. Expected Project Results

Based on the completed project execution:

### Logistic Regression

- Best C: 1
- Solver: liblinear
- Class Weight: None
- Cross-Validation ROC-AUC: approximately 0.9960

### Random Forest

- Number of estimators: 400
- Maximum depth: 10
- Minimum samples split: 2
- Minimum samples leaf: 4
- Maximum features: sqrt
- Class Weight: None
- Cross-Validation ROC-AUC: approximately 0.9901

The optimized Logistic Regression model achieved the strongest
cross-validation ROC-AUC among the optimized models.


## 8. Optimization Conclusion

Hyperparameter optimization improved the experimental understanding
of the models and identified configurations that provide strong
generalization performance.

The optimized models can now be evaluated on the untouched test set
for final project reporting.
